# Whole Census Geocoder Analysis

This notebook analyzes match performance across all chunked outputs from the whole-dataset geocoder and includes a US state map of match rates.

In [8]:
import glob
import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

DATA_GLOB = "../../data/1_derived/3_whole_properties_census_geocoded_*.csv"
files = sorted(glob.glob(DATA_GLOB))
if not files:
    raise FileNotFoundError(f"No files found for pattern: {DATA_GLOB}")

print(f"Found {len(files)} geocoded chunk files")
for f in files:
    print(f" - {os.path.basename(f)}")

df = pd.concat((pd.read_csv(f, low_memory=False) for f in files), ignore_index=True)
print(f"\nCombined shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()

Found 11 geocoded chunk files
 - 3_whole_properties_census_geocoded_001.csv
 - 3_whole_properties_census_geocoded_002.csv
 - 3_whole_properties_census_geocoded_003.csv
 - 3_whole_properties_census_geocoded_004.csv
 - 3_whole_properties_census_geocoded_005.csv
 - 3_whole_properties_census_geocoded_006.csv
 - 3_whole_properties_census_geocoded_007.csv
 - 3_whole_properties_census_geocoded_008.csv
 - 3_whole_properties_census_geocoded_009.csv
 - 3_whole_properties_census_geocoded_010.csv
 - 3_whole_properties_census_geocoded_011.csv

Combined shape: 1,051,219 rows x 34 columns


,LeaseCompId,PropertyId,PropertyName,Tenant.Name,TenantIndustry,TenantNAICS,Market,Submarket,Address.DeliveryAddress,Suite,...,longitude,state_fips,county_fips,census_tract,census_block,geoid,tie_matched_addresses,tie_latitudes,tie_longitudes,tie_geoids
0,114028931.0,435558.0,Bldg A,Robert A Garrett,Construction,NaN,Atlanta,Cumberland/Galleria,4501 Circle 75 Pky SE,1170,...,NaN,0.0,0.0,0.0,0.0,0.000000e+00,NaN,NaN,NaN,NaN
1,113952192.0,445018.0,The Palisades Medical Office,Physio,Health Care and Social Assistance,"Offices of Physical, Occupational and Speech T...",Atlanta,Upper Buckhead,3200 Downwood Cir NW,350,...,-84.426522,13.0,121.0,9803.0,2002.0,1.312101e+14,NaN,NaN,NaN,NaN
2,114008333.0,440921.0,100 Edgewood,Georgia Council on Substance Abuse,Health Care and Social Assistance,Individual and Family Services,Atlanta,Downtown Atlanta,100 Edgewood Ave NE,540,...,-84.386374,13.0,121.0,11901.0,3012.0,1.312101e+14,NaN,NaN,NaN,NaN
3,113897878.0,442552.0,Buckhead West - Bldg 1888,NaN,NaN,NaN,Atlanta,Lower Buckhead,1888 Emery St NW,NaN,...,-84.414403,13.0,121.0,9001.0,2001.0,1.312101e+14,NaN,NaN,NaN,NaN
4,114173293.0,444958.0,One Alliance Center,Aspen Insurance,Finance and Insurance,Insurance Related Activities,Atlanta,Upper Buckhead,3500 Lenox Rd NE,1710,...,-84.360784,13.0,121.0,9606.0,1002.0,1.312101e+14,NaN,NaN,NaN,NaN


## Overall Match Performance

In [9]:
df["match"] = df.get("match", pd.Series(index=df.index, dtype="object")).fillna("Missing")

is_match = df["match"].eq("Match")
is_tie = df["match"].eq("Tie")
is_matched_or_tie = is_match | is_tie

overall = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Match rows",
            "Tie rows",
            "Matched-or-Tie rows",
            "Strict match rate (%)",
            "Matched-or-Tie rate (%)",
        ],
        "value": [
            len(df),
            int(is_match.sum()),
            int(is_tie.sum()),
            int(is_matched_or_tie.sum()),
            round(is_match.mean() * 100, 2),
            round(is_matched_or_tie.mean() * 100, 2),
        ],
    }
)

match_dist = (
    df["match"]
    .value_counts(dropna=False)
    .rename_axis("match")
    .reset_index(name="count")
)
match_dist["pct"] = (match_dist["count"] / len(df) * 100).round(2)

display(overall)
display(match_dist)

,metric,value
0,Total rows,1051219.00
1,Match rows,970482.00
2,Tie rows,6697.00
3,Matched-or-Tie rows,977179.00
4,Strict match rate (%),92.32
5,Matched-or-Tie rate (%),92.96


,match,count,pct
0,Match,970482,92.32
1,No_Match,73355,6.98
2,Tie,6697,0.64
3,Missing,685,0.07


## State-Level Match Rates and US Map

In [ ]:
fips_to_abbrev = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT",
    "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI", "16": "ID", "17": "IL",
    "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME", "24": "MD",
    "25": "MA", "26": "MI", "27": "MN", "28": "MS", "29": "MO", "30": "MT", "31": "NE",
    "32": "NV", "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA", "54": "WV",
    "55": "WI", "56": "WY", "60": "AS", "66": "GU", "69": "MP", "72": "PR", "78": "VI"
}

if "state_fips" in df.columns:
    state_fips_clean = (
        pd.to_numeric(df["state_fips"], errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(2)
    )
else:
    state_fips_clean = pd.Series([""] * len(df), index=df.index)

state_from_fips = state_fips_clean.map(fips_to_abbrev)

if "full_address_std" in df.columns:
    state_from_address = df["full_address_std"].astype(str).str.extract(
        r",\s*([A-Z]{2})\s+\d{5}(?:-\d{4})?(?:,|$)", expand=False
    )
else:
    state_from_address = pd.Series([pd.NA] * len(df), index=df.index, dtype="string")

df_state = df.copy()
state_abbrev = state_from_fips.combine_first(state_from_address)
state_abbrev = pd.Series(state_abbrev, index=df.index, dtype="string").str.strip().str.upper()
state_abbrev = state_abbrev.where(state_abbrev.str.match(r"^[A-Z]{2}$", na=False), pd.NA)
df_state["state_abbrev"] = state_abbrev

state_summary = (
    df_state.dropna(subset=["state_abbrev"])
    .groupby("state_abbrev", as_index=False)
    .agg(
        total_rows=("state_abbrev", "size"),
        match_rows=("match", lambda s: (s == "Match").sum()),
        tie_rows=("match", lambda s: (s == "Tie").sum()),
    )
)

for c in ["total_rows", "match_rows", "tie_rows"]:
    state_summary[c] = pd.to_numeric(state_summary[c], errors="coerce").fillna(0)

state_summary["strict_match_rate_pct"] = (
    state_summary["match_rows"] / state_summary["total_rows"] * 100
).round(2)
state_summary["match_or_tie_rate_pct"] = (
    (state_summary["match_rows"] + state_summary["tie_rows"]) / state_summary["total_rows"] * 100
).round(2)
state_summary = state_summary.sort_values(
    ["strict_match_rate_pct", "total_rows"], ascending=[False, False]
)

fig = px.choropleth(
    state_summary,
    locations="state_abbrev",
    locationmode="USA-states",
    color="strict_match_rate_pct",
    hover_name="state_abbrev",
    hover_data={
        "total_rows": ":,",
        "match_rows": ":,",
        "tie_rows": ":,",
        "strict_match_rate_pct": ":.2f",
        "match_or_tie_rate_pct": ":.2f",
    },
    scope="usa",
    color_continuous_scale="YlGnBu",
    title="Strict Census Match Rate by State (%)",
)
fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
fig.show()

state_summary.head(15)

## Data Quality by Match Status

In [ ]:
quality_cols = [c for c in ["latitude", "longitude", "geoid"] if c in df.columns]

quality_by_match = (
    df.groupby("match", dropna=False)[quality_cols]
    .apply(lambda x: x.notna().mean() * 100)
    .round(2)
    .reset_index()
)

overall_missing = pd.DataFrame({
    "column": quality_cols,
    "missing_count": [df[c].isna().sum() for c in quality_cols],
    "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in quality_cols],
})

display(quality_by_match)
display(overall_missing)

(      match  latitude  longitude  geoid
 0     Match     100.0      100.0  100.0
 1   Missing       0.0        0.0    0.0
 2  No_Match       0.0        0.0  100.0
 3       Tie     100.0      100.0  100.0,
       column  missing_count  missing_pct
 0   latitude          74040         7.04
 1  longitude          74040         7.04
 2      geoid            685         0.07)

## High-Volume County Patterns

In [ ]:
county_cols_ok = all(c in df.columns for c in ["state_fips", "county_fips", "match"])
if county_cols_ok:
    county_df = df.copy()
    county_df["state_fips"] = pd.to_numeric(county_df["state_fips"], errors="coerce").astype("Int64").astype(str).str.replace("<NA>", "", regex=False).str.zfill(2)
    county_df["county_fips"] = pd.to_numeric(county_df["county_fips"], errors="coerce").astype("Int64").astype(str).str.replace("<NA>", "", regex=False).str.zfill(3)
    county_df["county_geoid"] = county_df["state_fips"] + county_df["county_fips"]
    county_df = county_df[county_df["county_geoid"].str.len() == 5]

    county_summary = (
        county_df.groupby("county_geoid", as_index=False)
        .agg(
            total_rows=("county_geoid", "size"),
            match_rows=("match", lambda s: (s == "Match").sum()),
            tie_rows=("match", lambda s: (s == "Tie").sum()),
        )
    )
    county_summary["strict_match_rate_pct"] = (county_summary["match_rows"] / county_summary["total_rows"] * 100).round(2)
    county_summary["state_abbrev"] = county_summary["county_geoid"].str[:2].map(fips_to_abbrev)

    top_volume = county_summary.sort_values("total_rows", ascending=False).head(20)
    top_volume
else:
    print("County-level columns were not found in the dataset.")

## Tie Ambiguity Diagnostics

In [ ]:
tie_diag_cols = [c for c in ["PropertyId", "full_address_std", "tie_geoids", "tie_matched_addresses"] if c in df.columns]

tie_df = df[df["match"] == "Tie"].copy()
if not tie_df.empty and "tie_geoids" in tie_df.columns:
    tie_df["n_tie_geoids"] = tie_df["tie_geoids"].fillna("").astype(str).apply(
        lambda s: len({x.strip() for x in s.split("|") if x.strip()})
    )

    tie_summary = tie_df["n_tie_geoids"].value_counts().sort_index().rename_axis("candidate_geoids").reset_index(name="row_count")
    tie_summary["pct_of_ties"] = (tie_summary["row_count"] / len(tie_df) * 100).round(2)
    display_cols = tie_diag_cols + ["n_tie_geoids"]

    display(tie_summary)
    display(tie_df[display_cols].sort_values("n_tie_geoids", ascending=False).head(25))
else:
    print("No Tie rows or tie_geoids column unavailable.")

## Export Analysis Outputs

In [ ]:
OUT_DIR = "../../output/2_analysis"
TABLES_DIR = os.path.join(OUT_DIR, "tables")
FIGURES_DIR = os.path.join(OUT_DIR, "figures")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

overall.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_overall_summary.csv"), index=False)
match_dist.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_match_distribution.csv"), index=False)
state_summary.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_state_summary.csv"), index=False)
overall_missing.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_missing_summary.csv"), index=False)

if 'county_summary' in locals():
    county_summary.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_county_summary.csv"), index=False)

if 'tie_summary' in locals():
    tie_summary.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_tie_summary.csv"), index=False)

fig.write_html(
    os.path.join(FIGURES_DIR, "whole_geocoder_state_match_rate_map.html"),
    include_plotlyjs="cdn"
 )
print(f"Saved tables to: {TABLES_DIR}")
print(f"Saved figures to: {FIGURES_DIR}")

Saved outputs to: ../../output/2_analysis
